# Practical 7: Dog Breed Classification using Transfer Learning

**Problem Statement:** Develop an image classifier to identify dog breeds using pretrained CNN architectures.

**Activities:**
1. Use VGG16, ResNet, or MobileNet
2. Fine-tune pretrained model
3. Evaluate classification accuracy

**Dataset:** Stanford Dogs — 120 dog breeds, loaded via TensorFlow Datasets.

**Note:** Enable a GPU runtime (Runtime -> Change runtime type -> GPU) before running this notebook.

## 1. Import Libraries and Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras

(train_ds, test_ds), ds_info = tfds.load(
    'stanford_dogs',
    split=['train', 'test'],
    with_info=True,
    as_supervised=True
)

class_names = ds_info.features['label'].names
num_classes = len(class_names)
print("Number of classes:", num_classes)
print("Train examples:", ds_info.splits['train'].num_examples)
print("Test examples:", ds_info.splits['test'].num_examples)

In [ ]:
def clean_name(name):
    return name.split('-', 1)[1].replace('_', ' ')

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, (image, label) in enumerate(train_ds.take(5)):
    axes[i].imshow(image.numpy())
    axes[i].set_title(clean_name(class_names[label.numpy()]), fontsize=9)
    axes[i].axis('off')
plt.show()

## 2. Data Preprocessing

Images are resized to 160x160 (MobileNetV2's expected input size) and pixel values are rescaled to the [-1, 1] range expected by the pretrained network.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Build Transfer Learning Model

MobileNetV2, pretrained on ImageNet, is used as a frozen feature extractor. A new classification head is added on top and trained for the dog breed task.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = keras.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.summary()

## 4. Train the Classification Head

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_head = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

## 5. Fine-Tune the Pretrained Model

The top layers of the base network are unfrozen and trained with a low learning rate, allowing the pretrained features to adapt slightly to dog breed images.

In [ ]:
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fine_tune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

## 6. Evaluate Classification Accuracy

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
acc = history_head.history['accuracy'] + history_fine_tune.history['accuracy']
val_acc = history_head.history['val_accuracy'] + history_fine_tune.history['val_accuracy']

plt.figure(figsize=(8, 5))
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.axvline(x=len(history_head.history['accuracy']) - 1, linestyle='--', color='gray', label='Fine-tuning start')
plt.title('Accuracy: Head Training vs Fine-Tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 7. Display Prediction Results

In [ ]:
images, labels = next(iter(test_ds.take(1)))
predictions = model.predict(images, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)

display_images = ((images.numpy() + 1) * 127.5).astype('uint8')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(display_images[i])
    true_name = clean_name(class_names[labels[i].numpy()])
    pred_name = clean_name(class_names[predicted_labels[i]])
    color = 'green' if predicted_labels[i] == labels[i].numpy() else 'red'
    ax.set_title(f"Pred: {pred_name}\nTrue: {true_name}", color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Conclusion

In this practical, we:
- Loaded the Stanford Dogs dataset via TensorFlow Datasets
- Built a transfer learning model using pretrained MobileNetV2 as a frozen feature extractor
- Trained a new classification head for dog breed recognition
- Fine-tuned the top layers of the pretrained network
- Evaluated classification accuracy and displayed sample predictions